In [40]:
!pip -q install -U scikit-learn

import os
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_auc_score, average_precision_score
)

In [41]:
TFIDF_CSV = "/content/HDFS_tfidf_full.csv"
LABEL_CSV = "/content/anomaly_label.csv"

for p in [TFIDF_CSV, LABEL_CSV]:
    assert os.path.exists(p), f"Missing: {p}"
print("Files found.")

Files found.


In [42]:
tfidf_df = pd.read_csv(TFIDF_CSV)
tfidf_df.columns = [c.strip() for c in tfidf_df.columns]

lab_df = pd.read_csv(LABEL_CSV)
lab_df.columns = [c.strip() for c in lab_df.columns]

print("TFIDF shape:", tfidf_df.shape)
print("LABEL shape:", lab_df.shape)
print("LABEL columns:", lab_df.columns.tolist())

TFIDF shape: (575061, 49)
LABEL shape: (575061, 2)
LABEL columns: ['BlockId', 'Label']


In [43]:
ID_CANDIDATES = ["blockid","block_id","block","blk_id","BlockId","BlockID","id","Id"]
LABEL_CANDIDATES = ["label","labels","anomaly","is_anomaly","y","target","class"]

def detect_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

JOIN_COL  = detect_col(lab_df, ID_CANDIDATES)
LABEL_COL = detect_col(lab_df, LABEL_CANDIDATES)

if JOIN_COL is None:
    raise ValueError(f"Could not detect BlockId column in label file. Columns: {lab_df.columns.tolist()}")
if LABEL_COL is None:
    raise ValueError(f"Could not detect label column in label file. Columns: {lab_df.columns.tolist()}")

print("Detected JOIN_COL:", JOIN_COL)
print("Detected LABEL_COL:", LABEL_COL)

merged = tfidf_df.merge(lab_df[[JOIN_COL, LABEL_COL]], on=JOIN_COL, how="inner")
print("Merged shape:", merged.shape)

Detected JOIN_COL: BlockId
Detected LABEL_COL: Label
Merged shape: (575061, 50)


In [44]:
def make_xy(df: pd.DataFrame, join_col: str, label_col: str):
    y_raw = df[label_col]

    if pd.api.types.is_numeric_dtype(y_raw):
        y = (y_raw.astype(float) > 0).astype(int).values
    else:
        y_str = y_raw.astype(str).str.lower()
        y = y_str.isin(["1","true","yes","y","anomaly","abnormal"]).astype(int).values

    X = df.drop(columns=[join_col, label_col])

    # force numeric
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.fillna(0.0)

    return X.values.astype(np.float32), y

X, y = make_xy(merged, JOIN_COL, LABEL_COL)
print("X:", X.shape, "y:", y.shape, "anomaly_rate:", y.mean())

X: (575061, 48) y: (575061,) anomaly_rate: 0.029280371995318757


In [45]:
USE_SUBSET = True
SUBSET_N = 25000
RANDOM_STATE = 42

if USE_SUBSET and X.shape[0] > SUBSET_N:
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(np.arange(X.shape[0]), size=SUBSET_N, replace=False)
    idx = np.sort(idx)

    merged = merged.iloc[idx].reset_index(drop=True)
    X, y = make_xy(merged, JOIN_COL, LABEL_COL)

print("After subset -> X:", X.shape, "anomaly_rate:", y.mean())

After subset -> X: (25000, 48) anomaly_rate: 0.02968


In [46]:
N_COMPONENTS = 30  # try 50 or 100 (30 can be too aggressive)

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
Z = svd.fit_transform(X)

scaler = StandardScaler(with_mean=True, with_std=True)
Zs = scaler.fit_transform(Z)

print("Embedding shape:", Zs.shape)
print("Explained variance sum:", float(svd.explained_variance_ratio_.sum()))

Embedding shape: (25000, 30)
Explained variance sum: 0.9999154210090637


In [47]:
def eval_predictions(y_true, y_pred, scores=None, title=""):
    cm = confusion_matrix(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

    print("\n" + "="*90)
    print(title)
    print("Confusion matrix [[TN FP],[FN TP]]:\n", cm)
    print(f"Precision={pr:.4f}  Recall={rc:.4f}  F1={f1:.4f}")

    if scores is not None:
        try:
            auc = roc_auc_score(y_true, scores)
            ap = average_precision_score(y_true, scores)
            print(f"ROC-AUC={auc:.4f}  PR-AUC(AP)={ap:.4f}")
        except Exception as e:
            print("AUC/AP not computed:", e)

    print("\nClassification report:\n", classification_report(y_true, y_pred, digits=4, zero_division=0))

In [48]:
def dbscan_fit_and_score(Zs, eps, min_samples):
    db = DBSCAN(eps=float(eps), min_samples=int(min_samples), metric="euclidean", n_jobs=-1)
    labels = db.fit_predict(Zs)

    # Core points
    core_idx = getattr(db, "core_sample_indices_", None)
    if core_idx is None or len(core_idx) == 0:
        # If DBSCAN produced no core points, return worst-case scores
        scores = np.ones(Zs.shape[0], dtype=np.float32)
        return db, labels, scores

    core = Zs[core_idx]

    # Distance to nearest core point = anomaly score (higher => more anomalous)
    nn = NearestNeighbors(n_neighbors=1, metric="euclidean", n_jobs=-1)
    nn.fit(core)
    dist, _ = nn.kneighbors(Zs)
    scores = dist.ravel().astype(np.float32)

    return db, labels, scores

In [49]:
min_samples_grid = [5, 10, 20, 30]
# eps candidates from k-distance percentiles (computed per min_samples)
eps_percentiles = [90, 93, 95, 97, 98, 99]
# decision threshold on score (percentiles)
score_percentiles = [90, 93, 95, 97, 98, 99]

best = {"f1": -1}

for ms in min_samples_grid:
    nn = NearestNeighbors(n_neighbors=ms, metric="euclidean", n_jobs=-1).fit(Zs)
    d, _ = nn.kneighbors(Zs)
    kdist = np.sort(d[:, -1])

    for ep in eps_percentiles:
        eps = float(np.percentile(kdist, ep))
        # Ensure eps is always greater than 0, as required by DBSCAN
        eps = max(np.finfo(float).eps, eps)
        db, labels, scores = dbscan_fit_and_score(Zs, eps=eps, min_samples=ms)

        # sweep score threshold
        for sp in score_percentiles:
            thr = float(np.percentile(scores, sp))
            y_pred = (scores >= thr).astype(int)

            pr, rc, f1, _ = precision_recall_fscore_support(y, y_pred, average="binary", zero_division=0)

            if f1 > best["f1"]:
                best = {
                    "f1": float(f1),
                    "precision": float(pr),
                    "recall": float(rc),
                    "min_samples": int(ms),
                    "eps_percentile": int(ep),
                    "eps": float(eps),
                    "score_percentile": int(sp),
                    "score_threshold": float(thr),
                    "pred_rate": float(y_pred.mean()),
                    "noise_rate": float(np.mean(labels == -1)),
                    "clusters": int(len(set(labels)) - (1 if -1 in labels else 0))
                }

print("BEST CONFIG:\n", best)


BEST CONFIG:
 {'f1': 0.7306501547987616, 'precision': 0.8581818181818182, 'recall': 0.6361185983827493, 'min_samples': 20, 'eps_percentile': 99, 'eps': 2.7170302867889404, 'score_percentile': 98, 'score_threshold': 3.3717478231665154e-07, 'pred_rate': 0.022, 'noise_rate': 0.0084, 'clusters': 16}


In [50]:
db, labels, scores = dbscan_fit_and_score(
    Zs,
    eps=best["eps"],
    min_samples=best["min_samples"]
)

y_pred = (scores >= best["score_threshold"]).astype(int)

eval_predictions(
    y_true=y,
    y_pred=y_pred,
    scores=scores,
    title=f"DBSCAN tuned | eps={best['eps']:.4f} (P{best['eps_percentile']}) "
          f"min_samples={best['min_samples']} | score_thr=P{best['score_percentile']}"
)

print("Clusters (excluding noise):", best["clusters"])
print("DBSCAN noise rate (labels==-1):", best["noise_rate"])
print("Predicted anomaly rate (after score threshold):", best["pred_rate"])


DBSCAN tuned | eps=2.7170 (P99) min_samples=20 | score_thr=P98
Confusion matrix [[TN FP],[FN TP]]:
 [[24180    78]
 [  270   472]]
Precision=0.8582  Recall=0.6361  F1=0.7307
ROC-AUC=0.8106  PR-AUC(AP)=0.5774

Classification report:
               precision    recall  f1-score   support

           0     0.9890    0.9968    0.9929     24258
           1     0.8582    0.6361    0.7307       742

    accuracy                         0.9861     25000
   macro avg     0.9236    0.8165    0.8618     25000
weighted avg     0.9851    0.9861    0.9851     25000

Clusters (excluding noise): 16
DBSCAN noise rate (labels==-1): 0.0084
Predicted anomaly rate (after score threshold): 0.022


In [52]:
# =========================
# EXPORT DBSCAN RESULTS
# =========================

OUT_PATH = "/content/dbscan_results.csv"

out = pd.DataFrame({
    "BlockId": merged[JOIN_COL].astype(str).values,
    "y_true": y.astype(int),
    "score_dbscan": scores.astype(float),
    "pred_dbscan": y_pred.astype(int)
})

out.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
out.head()

Saved: /content/dbscan_results.csv


,BlockId,y_true,score_dbscan,pred_dbscan
0,blk_-3544583377289625738,1,3.089015,1
1,blk_-4513615671112005170,0,0.000000,0
2,blk_-1385756122847916710,0,0.000000,0
3,blk_-3667585438930153850,0,0.000000,0
4,blk_3522654861930890662,0,0.000000,0
